# 04 - Trading simulations and deep analysis

Rubric: *Trading simulation (8)*. Test period: 2022 onward, walk-forward predictions only.

**Vector simulations** (`sim1_` ... `sim6_` columns): THB 100 per signal, buy next open, sell close 5 days later,
0.15% cost per side.

**Exact simulations** (day-by-day loop, THB 1,000,000, profits reinvested, 100-share lots, weekly rebalance):

| Strategy | Weights | Risk management |
| --- | --- | --- |
| E1 | equal weight, top-10 ensemble scores | none |
| E2 | equal weight, top-10, keep holdings while they stay in the top-30 (turnover buffer) | 8% stop-loss on the daily low |
| E3 | mean-variance on Grinold alphas + Ledoit-Wolf risk (lambda = 1, 25% cap) | 8% stop-loss |
| E4 | E3 | + halve exposure when HMM P(stressed) > 0.5 or VIX > VIX3M |
| E5 | E3 | + short the SET50 ETF against the book's beta (long-short, market-neutral) |
| E6 | conservative mean-variance (lambda = 5, 15% cap) | 15% volatility target + stop |

**Selection protocol:** the strategy marked `*` is chosen automatically *before* looking at the test years: it is the
one with the highest Sharpe on the 2020-2021 validation years. The test table then shows how it did out of sample. The Deflated Sharpe Ratio (DSR)
accounts for having tried six strategies.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from setalpha import config, ingest, transform, model, regime, portfolio, strategy, evaluate, report, pipeline
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True})

In [ ]:
res = joblib.load(pipeline.SIM_RESULTS)
res["vector"].round(2)

In [ ]:
print("VALIDATION 2020-2021 (used to choose the strategy)")
display(res["metrics_valid"][["selected", "CAGR %", "Sharpe", "max DD %", "info ratio", "alpha % p.a."]].round(3))
print("TEST 2022 onward (out of sample): factsheet")
for group, table in evaluate.grouped(res["metrics"]).items():
    print(f"\n--- {group} ---"); display(table.round(2))

## Factor attribution (Fama-French / Carhart)

* **Carhart 4-factor on the Fama-French Emerging Markets factors** (Ken French library, monthly, USD; strategy
  returns converted from THB with USD/THB).
* **Local Thai factors** (daily, THB): MKT = SET50 ETF, SMB = small minus big liquid stocks, WML = 12-1 month
  momentum. A significant positive SMB loading means the book is tilted to smaller stocks.

In [ ]:
print("Carhart 4-factor, Fama-French Emerging Markets (monthly, USD)"); display(res["factors_em"].round(3))
print("Local Thai factors (daily, THB)"); display(res["factors_th"].round(3))
lf = res["local_factors"]
print("Annualised local factor returns, test period:", (lf[lf.index >= "2022"].mean() * 252 * 100).round(2).to_dict())
(1 + lf[lf.index >= "2013"]).cumprod().plot(logy=True, title="Local Thai factors, cumulative (MKT = SET50 ETF)", figsize=(13, 4));

In [ ]:
eq = (1 + res["daily_returns"].fillna(0)).cumprod()
eq.plot(logy=True, title="Equity curves, start = 1 (log scale)", figsize=(13, 5));
dd = eq / eq.cummax() - 1
dd.mul(100).plot(title="Drawdown %", figsize=(13, 3.5));

In [ ]:
roll = evaluate.rolling_return(res["daily_returns"].dropna(how="all").fillna(0))
roll.mul(100).plot(title="Rolling 12-month return %", figsize=(13, 3.5));
res["exposure"].rolling(20).mean().mul(100).plot(title="Gross long exposure % (20d avg)", figsize=(13, 3));

## Deep analysis: where do the predictions work?

Top-decile picks of the ensemble on the test years, split by sector, year, HMM regime, liquidity quintile and month.
`lift` is precision minus the base rate in the same group.

In [ ]:
for k in ["by_sector", "by_year", "by_regime", "by_size", "by_month"]:
    print(f"\n{k}"); display(res[k].round(2))

## Debugging the worst confident calls

In [ ]:
res["worst_calls"]

In [ ]:
print("selected strategy:", res["selected"])
trades = res["trades"][res["selected"]]
print(trades["reason"].value_counts())
trades.tail(10)